In [16]:
import glob
import pandas as pd
import numpy as np

def read_mat_data(filepath):
    import scipy.io
    rd = scipy.io.loadmat(filepath) # load the mat file
    PDATA = rd['PDATA']
    TWT = rd['TWT']
    
    return PDATA, TWT

In [17]:
# load TWT vector
[PDATA, TWT] = read_mat_data('/Users/jukesliu/Documents/POSTDOC/snow-radar/ReynoldsMountain/ProcessedMay25_xyz/PDATA/pdriftzigzag_8_PDATA.mat')
TWT = TWT.flatten()
TWT = TWT[int(len(TWT)/2):]
TWT.shape

(8192,)

In [18]:
TWT_interval = np.diff(TWT)[0]
print(TWT_interval)

1.1418660481770834e-11


# DEPTHS GOT MESSED UP, ADJUST

In [79]:
# grab CSV files
folderpath = '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/'
files = glob.glob(folderpath+'*')
files.sort()
files

['/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd01_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd02_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd03_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd04_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd05_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd06_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd07_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd08_1000scale_100size_1mod_filtered.csv',
 '/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered

In [87]:
for file in files:
    file_df = pd.read_csv(file) # read in original file
    print(file)
    if not file.startswith('rd_'):
        if 'rd_id' not in list(file_df.columns):
            # add rd_id column
            rd_id_col = []
            for j in range(0, len(file_df)):
                rd_id_col.append(file.split('/')[-1].split('_')[0]) # append the rd_id from the file path
            file_df['rd_id'] = rd_id_col
    
        # add in TWT
        TWT_surf = TWT[(file_df.Isurf).astype(int)]
        TWT_ground = TWT[(file_df.Iground).astype(int)]
        file_df['TWT_surf'] = TWT_surf; file_df['TWT_ground'] = TWT_ground
    
        # cleaned_df = file_df[["rd_id", "x", "y", "Isurf", "Iground", "TWT_surf", "TWT_ground","x_idx", "depth"]] # grab the relevant subset
    
        # # rename certain columns
        # cleaned_df = cleaned_df.rename(columns={'x': 'UTMx', 'y': 'UTMy', 'x_idx': 'rd_xidx', 
        #                                         'depth': 'radar_depth_cm'})

    
        # round the radar depth cm to the nearest integer
        cleaned_df['radar_depth_cm'] = np.round(cleaned_df['radar_depth_cm']).astype(int)

        # add depth shift!
        cleaned_df.radar_depth_cm = cleaned_df.radar_depth_cm -41
    
        # overwrite
        cleaned_df.to_csv(file)

/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd01_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd02_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd03_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd04_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd05_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd06_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd07_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd08_1000scale_100size_1mod_filtered.csv
/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/filtered/rd09_1000scale_100size_1mod_filte

In [88]:
all_dfs = []
for file in files:
    file_df = pd.read_csv(file) # read in original file
    all_dfs.append(file_df)

all_df = pd.concat(all_dfs,ignore_index=True)
all_df = all_df.drop(['Unnamed: 0'],axis=1)
all_df.to_csv(folderpath+'rd_picks_1000scale_100size_1mod_filtered.csv')

# Clean up CSV file

In [10]:
file_df = pd.read_csv('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/all_rds_lidar0removed.csv')
file_df = file_df.drop(columns='Unnamed: 0')
file_df

# shift the ground index


# # add in TWT
# TWT_surf = TWT[(file_df.Isurf).astype(int)]
# TWT_ground = TWT[(file_df.Iground).astype(int)]
# file_df['TWT_surf'] = TWT_surf; file_df['TWT_ground'] = TWT_ground

# cleaned_df = file_df[["rd_id", "x", "y", "Isurf", "Iground", "TWT_surf", "TWT_ground","x_idx", "depth"]] # grab the relevant subset

# # rename certain columns
# cleaned_df = cleaned_df.rename(columns={'x': 'UTMx', 'y': 'UTMy', 'x_idx': 'rd_xidx', 
#                                         'depth': 'radar_depth_cm'})

# # round the radar depth cm to the nearest integer
# cleaned_df['radar_depth_cm'] = np.round(cleaned_df['radar_depth_cm']).astype(int)

# # add depth shift!
# cleaned_df.radar_depth_cm = cleaned_df.radar_depth_cm - 41

# # overwrite
# cleaned_df.to_csv('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/all_rds_lidar0removed.csv')
# cleaned_df

,rd_id,UTMx,UTMy,Isurf,Iground,TWT_surf,TWT_ground,rd_xidx,radar_depth_cm
0,rd01,519867.619504,4.768088e+06,539.0,1754.0,6.154658e-09,2.002833e-08,52.0,115
1,rd01,519867.614812,4.768088e+06,539.0,1756.0,6.154658e-09,2.005117e-08,53.0,115
2,rd01,519867.610043,4.768088e+06,539.0,1758.0,6.154658e-09,2.007401e-08,54.0,116
3,rd01,519867.605194,4.768088e+06,539.0,1759.0,6.154658e-09,2.008542e-08,55.0,116
4,rd01,519867.600276,4.768088e+06,539.0,1761.0,6.154658e-09,2.010826e-08,56.0,116
...,...,...,...,...,...,...,...,...,...
115222,rd45,519911.090432,4.768364e+06,485.0,1303.0,5.538050e-09,1.487851e-08,6652.0,64
115223,rd45,519911.104785,4.768364e+06,484.0,1301.0,5.526632e-09,1.485568e-08,6661.0,64
115224,rd45,519911.913311,4.768365e+06,475.0,1272.0,5.423864e-09,1.452454e-08,6981.0,61
115225,rd45,519911.916122,4.768365e+06,473.0,1261.0,5.401026e-09,1.439893e-08,7062.0,60


## Shift values based off of peak shift

41.53846153846155 m  =  323 pixels

In [21]:
depth_shift = 41.53846153846155 
velocity = 2.225e10 # cm/s
pixel_shift = round(depth_shift*2/velocity/TWT_interval)
print(pixel_shift)

327


In [30]:
file_df = pd.read_csv('/Users/jukesliu/Documents/POSTDOC/snow-radar/RY_output/coords/all_rds_lidar0removed.csv')
file_df = file_df.drop(columns='Unnamed: 0')

# shift the ground index
file_df.Iground = file_df.Iground-pixel_shift

# grab the new TWT ground
TWT_ground = TWT[(file_df.Iground).astype(int)]
file_df['TWT_ground'] = TWT_ground

# recalculate depths using new velocity
shifted_depths = (file_df.TWT_ground-file_df.TWT_surf)*velocity/2
file_df.radar_depth_cm = np.round(shifted_depths).astype(int)

# drop rd_idx
file_df = file_df.drop(columns='rd_xidx')
file_df

,rd_id,UTMx,UTMy,Isurf,Iground,TWT_surf,TWT_ground,radar_depth_cm
0,rd01,519867.619504,4.768088e+06,539.0,1427.0,6.154658e-09,1.629443e-08,113
1,rd01,519867.614812,4.768088e+06,539.0,1429.0,6.154658e-09,1.631727e-08,113
2,rd01,519867.610043,4.768088e+06,539.0,1431.0,6.154658e-09,1.634010e-08,113
3,rd01,519867.605194,4.768088e+06,539.0,1432.0,6.154658e-09,1.635152e-08,113
4,rd01,519867.600276,4.768088e+06,539.0,1434.0,6.154658e-09,1.637436e-08,114
...,...,...,...,...,...,...,...,...
115222,rd45,519911.090432,4.768364e+06,485.0,976.0,5.538050e-09,1.114461e-08,62
115223,rd45,519911.104785,4.768364e+06,484.0,974.0,5.526632e-09,1.112178e-08,62
115224,rd45,519911.913311,4.768365e+06,475.0,945.0,5.423864e-09,1.079063e-08,60
115225,rd45,519911.916122,4.768365e+06,473.0,934.0,5.401026e-09,1.066503e-08,59


In [31]:
# write to CSV
file_df.to_csv('/Users/jukesliu/Documents/POSTDOC/snow-radar/data_submission/sample_data_RME/rdpicks_1000scale_100size_1mod_noheader.csv')